# 03. Reward model

Stage 3 of `docs/codeagents/research_plan.md` (§5, §11 stage 3): a feedback model with heads `ratio`, `listen_plus`, `skip`, `like`, `dislike`, `continue`, its ensemble uncertainty, and its evaluation on the val / test day against train-mean baselines (plan §5.4).

- **proxy RM** (`rm_proxy`): encoder initialised from the base model `runs/base/best.pt` (run `02_baseline` first), 4 bootstrap heads. This is the reward for RL.
- **gold RM** (`rm_gold`): evaluation only. Different seed, random init (no base weights), 80% of train users, + audio embeddings (needs `embeddings.parquet` on Drive: `uv run yambda-download --embeddings`, ~13.8 GB; set `rm.audio_embeddings=false` otherwise). Proxy reward growing while gold reward does not is the over-optimisation signal (Gao et al., 2022).

Architecture and leak rules are in the `recsys_rl.reward_model` docstring: the prediction for listen *t* uses only listens < *t* (with their feedback) and features of *t* known at recommendation time. Logic lives in `recsys_rl.reward_model` and `recsys_rl.rm_eval`. The same training from a terminal: `uv run yambda-train-rm run_name=rm_proxy`. Tables and plots for the report go to `results/stage3/` on Drive (download that folder into the repo's `results/`).

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import torch
from scipy.stats import spearmanr

from recsys_rl import evaluate, rm_eval
from recsys_rl.config import load_config, parse_overrides
from recsys_rl.data import prepare
from recsys_rl.reward_model import (
    HEAD_INDEX,
    build_feedback_set,
    load_reward_model,
    predict,
    train_reward_model,
)
from recsys_rl.train import amp_dtype
from recsys_rl.utils import get_device, save_figure

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_float_precision(4)

device = get_device()
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")
cfg = load_config("configs/base.yaml")
prepared = prepare(cfg)
store = prepared.store.in_memory()
dtype = amp_dtype(device, cfg.rm.amp)
RESULTS = cfg.results_dir / "stage3"  # CSV / PNG for the report, on Drive in Colab
RESULTS.mkdir(parents=True, exist_ok=True)

## Training

Selection by the val loss (sum of per-head log losses of the ensemble mean). Training resumes from `runs/<run_name>/last.pt` after a disconnect.

In [ ]:
RUNS = {
    "rm_proxy": [],
    "rm_gold": [
        "seed=7",
        "rm.base_run=null",
        "rm.audio_embeddings=true",
        "rm.user_fraction=0.8",
    ],
}


def run(run_name: str, *overrides: str):
    run_cfg = load_config(
        "configs/base.yaml", **parse_overrides([f"run_name={run_name}", *overrides])
    )
    return load_reward_model(train_reward_model(run_cfg, prepared, device), device)


rms = {name: run(name, *overrides) for name, overrides in RUNS.items()}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in RUNS:
    log = pl.read_ndjson(cfg.run_dir.parent / name / "log.jsonl")
    for ax, col in zip(axes, ["val_loss", "val_ratio_mae", "val_like_pr_auc"], strict=True):
        ax.plot(log["step"], log[col], label=name)
        ax.set(xlabel="step", ylabel=col)
axes[0].legend()
save_figure(fig, RESULTS / "training.png")
plt.show()

## Quality vs baselines (val and test)

Baselines are train means of each target, per user / per item / additive user + item, shrunk towards the global mean (10 pseudo-listens). Reference from Matveev et al. on Yambda-5B: ratio MAE ≈ 0.177, like PR-AUC ≈ 0.24.

In [ ]:
sets, probs, preds = {}, {}, {}
for split in ("val", "test"):
    fs = build_feedback_set(
        store, prepared.bounds, split, cfg.data.max_seq_len, cfg.data, cfg.rm.skip_threshold
    )
    sets[split] = fs
    probs[split] = {name: predict(rm, store, fs, amp_dtype=dtype) for name, rm in rms.items()}
    preds[split] = {name: p.mean(0) for name, p in probs[split].items()}
    preds[split] |= rm_eval.baseline_predictions(
        store, prepared.bounds, fs, cfg.data, cfg.rm.skip_threshold
    )
quality = pl.concat([rm_eval.quality_table(preds[s], sets[s]) for s in sets])
quality.write_csv(RESULTS / "quality.csv")
quality.select(
    "split",
    "method",
    "events",
    "loss",
    "ratio_mae",
    "ratio_rmse",
    "ratio_spearman",
    "listen_plus_auc",
    "skip_auc",
    "like_auc",
    "like_pr_auc",
    "dislike_pr_auc",
    "continue_auc",
    "ratio_ece",
    "like_ece",
)

## Slices (test)

Track length, popularity (train listens), first listen vs repeat (whole history), organic vs recommendation, position in the session. Where the RM is much better than baselines is where it can be trusted; the mean prediction per slice shows which kinds of tracks it rates highly.

In [ ]:
fs = sets["test"]
slices = rm_eval.event_slices(store, prepared.items, fs, cfg.data)
by_slice = rm_eval.slice_table(
    {k: preds["test"][k] for k in ("rm_proxy", "rm_gold", "user+item")}, fs, slices
)
by_slice.write_csv(RESULTS / "slices.csv")
by_slice

## Calibration (test)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
calibs = []
for ax, head in zip(axes, ["ratio", "listen_plus", "like"], strict=True):
    calib = rm_eval.calibration_table(
        {k: preds["test"][k] for k in ("rm_proxy", "rm_gold", "user+item")}, fs, head, n_bins=20
    )
    calibs.append(calib)
    for method, part in calib.filter(pl.col("count") > 0).group_by("method", maintain_order=True):
        ax.plot(part["mean_pred"], part["mean_true"], marker="o", label=method[0])
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set(title=head, xlabel="mean prediction", ylabel="observed")
    if head == "like":
        ax.set(xscale="log", yscale="log")
axes[0].legend()
pl.concat(calibs).write_csv(RESULTS / "calibration.csv")
save_figure(fig, RESULTS / "calibration.png")
plt.show()

## Ensemble uncertainty and proxy–gold agreement (test)

If the ensemble spread is informative, the error of the mean prediction grows with it (then `r − λ·σ` in R5 makes sense). Proxy–gold rank correlation on logged listens is the baseline agreement; in RL we expect it to drop on the policy's own recommendations.

In [ ]:
uncertainty = rm_eval.uncertainty_table(probs["test"]["rm_proxy"], fs)
uncertainty.write_csv(RESULTS / "uncertainty.csv")
display(uncertainty)
agreement = []
for head in ("ratio", "like", "continue"):
    h = HEAD_INDEX[head]
    v = fs.valid[:, h]
    rho = spearmanr(preds["test"]["rm_proxy"][v, h], preds["test"]["rm_gold"][v, h]).statistic
    agreement.append({"head": head, "proxy_gold_spearman": float(rho)})
agreement = pl.DataFrame(agreement)
agreement.write_csv(RESULTS / "proxy_gold_agreement.csv")
agreement

## Sensitivity: cheap paths for RL

Mean prediction when one candidate feature is forced to each value for every test listen (the rest unchanged). A strong rise with `repeat_count` or with short `length_bin` is a direction RL can exploit with R1 (ratio); R4 (length-debiased ratio) and the repeat penalty of R5 target exactly these.

In [ ]:
sub = build_feedback_set(
    store,
    prepared.bounds,
    "test",
    cfg.data.max_seq_len,
    cfg.data,
    cfg.rm.skip_threshold,
    max_users=2000,
)
sens = pl.concat(
    [
        rm_eval.sensitivity(rm, store, sub, cfg.model.n_length_bins, amp_dtype=dtype).with_columns(
            rm=pl.lit(name)
        )
        for name, rm in rms.items()
    ]
)
sens.write_csv(RESULTS / "sensitivity.csv")
sens

## What the RM prefers over the whole catalogue

RL samples from the whole vocabulary, mostly items the user never heard. For 512 frozen test contexts every vocabulary item is scored as a recommendation (`is_organic = 0`): mean score by length / popularity, and the composition of the RM's top-100 vs the catalogue. This is what a pure reward maximiser would recommend.

In [ ]:
es = evaluate.build_eval_set(
    store, prepared.bounds, "test", cfg.data.max_seq_len, cfg.data.listen_plus_threshold
)


def quintile(col: str) -> pl.Expr:
    """q1..q5 by rank (q1: shortest / least popular); robust to tied values."""
    q = (pl.col(col).rank("ordinal") - 1) * 5 // pl.len() + 1
    return pl.format("q{}", q)


by_group, top_summary = [], []
for name, rm in rms.items():
    per_item, top_k = rm_eval.catalog_profile(rm, store, prepared.items, es, amp_dtype=dtype)
    per_item = per_item.with_columns(
        length=quintile("track_length"),
        popularity=quintile("count"),
    )
    for group in ("length", "popularity"):
        by_group.append(
            per_item.group_by(group)
            .agg(items=pl.len(), mean_score=pl.col("mean_score").mean())
            .sort(group)
            .rename({group: "value"})
            .with_columns(rm=pl.lit(name), group=pl.lit(group))
        )
    top_summary.append(top_k.mean().with_columns(rm=pl.lit(name)))
catalog = pl.concat(by_group).select("rm", "group", "value", "items", "mean_score")
catalog.write_csv(RESULTS / "catalog_scores.csv")
top_summary = pl.concat(top_summary)
top_summary.write_csv(RESULTS / "catalog_top100.csv")
display(catalog)
top_summary

## Findings

To be filled after the Colab run: RM vs baselines (overall and per slice), calibration, usefulness of the ensemble spread, proxy–gold agreement, and the cheap paths to a high reward (short tracks, repeats, popular items) that RL is expected to exploit.